### Define Spoken Digits

In [2]:
from utils.homophones import get_homophones
from utils.rhymes import get_rhymes

# Base spoken-to-digit mapping (only the core numbers)
base_spoken_to_digit = {
    "zero": "0",
    "one": "1",
    "two": "2",
    "three": "3",
    "four": "4",
    "five": "5",
    "six": "6",
    "seven": "7",
    "eight": "8",
    "nine": "9",
}


# Function to expand the dictionary using homophones and rhymes
def expand_spoken_numbers():
    expanded_spoken_to_digit = base_spoken_to_digit.copy()

    for word, digit in base_spoken_to_digit.items():
        try:
            # Fetch homophones and rhymes
            homophones = get_homophones(word)
            rhymes = get_rhymes(word)

            # Add variations to dictionary
            for variation in homophones + rhymes:
                if variation.lower() not in expanded_spoken_to_digit:
                    expanded_spoken_to_digit[variation.lower()] = digit

        except Exception as e:
            print(f"Error fetching homophones/rhymes for '{word}': {e}")

    return expanded_spoken_to_digit


# Run locally to expand your dictionary
expanded_spoken_to_digit = expand_spoken_numbers()

# Print expanded mapping (optional)
print(expanded_spoken_to_digit)


{'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4', 'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9', 'xero-': '0', '-xero': '0', 'hero': '0', 'nero': '0', 'superhero': '0', 'nonexistents': '1', 'confectionary': '1', 'thrown': '1', 'none': '1', 'done': '1', 'won': '1', 'kernel': '1', 'confectionery': '1', 'bologna': '1', 'loner': '1', 'stationery': '1', 'dun': '1', 'sown': '1', 'barrenness': '1', 'baroness': '1', 'krone': '1', 'colonel': '1', 'shone': '1', 'lone': '1', 'baloney': '1', 'loan': '1', 'sewn': '1', 'nun': '1', 'nonexistence': '1', 'krona': '1', 'sone': '1', 'shown': '1', 'loaner': '1', 'stationary': '1', 'throne': '1', 'run': '1', 'gun': '1', 'fun': '1', 'to': '2', 'too': '2', 'through': '2', 'blue': '2', 'tree': '3', 'be': '3', 'sea': '3', 'fourth': '4', 'for': '4', 'forth': '4', 'fore': '4', 'war': '4', 'door': '4', 'drive': '5', 'dive': '5', 'strive': '5', 'sics': '6', 'fix': '6', 'politics': '6', 'mix': '6', 'heaven': '7', 'leaven': '7', 'le

In [1]:
spoken_to_digit = {
    "zero": "0",
    "one": "1",
    "two": "2",
    "three": "3",
    "four": "4",
    "five": "5",
    "six": "6",
    "seven": "7",
    "eight": "8",
    "nine": "9",
    "xero-": "0",
    "-xero": "0",
    "hero": "0",
    "nero": "0",
    "superhero": "0",
    "dun": "1",
    "nun": "1",
    "none": "1",
    "kernel": "1",
    "loaner": "1",
    "shone": "1",
    "stationery": "1",
    "done": "1",
    "sone": "1",
    "shown": "1",
    "sown": "1",
    "baroness": "1",
    "colonel": "1",
    "baloney": "1",
    "confectionery": "1",
    "sewn": "1",
    "krona": "1",
    "nonexistents": "1",
    "nonexistence": "1",
    "loner": "1",
    "won": "1",
    "bologna": "1",
    "lone": "1",
    "barrenness": "1",
    "throne": "1",
    "confectionary": "1",
    "krone": "1",
    "thrown": "1",
    "stationary": "1",
    "loan": "1",
    "run": "1",
    "gun": "1",
    "fun": "1",
    "to": "2",
    "too": "2",
    "through": "2",
    "blue": "2",
    "tree": "3",
    "be": "3",
    "sea": "3",
    "fourth": "4",
    "forth": "4",
    "fore": "4",
    "for": "4",
    "war": "4",
    "door": "4",
    "drive": "5",
    "dive": "5",
    "strive": "5",
    "sics": "6",
    "fix": "6",
    "politics": "6",
    "mix": "6",
    "heaven": "7",
    "leaven": "7",
    "levin": "7",
    "weighted": "8",
    "waits": "8",
    "hight": "8",
    "ate": "8",
    "waited": "8",
    "wait": "8",
    "weighting": "8",
    "height": "8",
    "slight": "8",
    "weight": "8",
    "waiting": "8",
    "weights": "8",
    "sleight": "8",
    "ait": "8",
    "state": "8",
    "rate": "8",
    "date": "8",
    "canine": "9",
    "k9": "9",
    "line": "9",
    "design": "9",
    "sign": "9",
}

### Extract Flight Details from Sentence via Fuzzy Matching
- Levenshtein distance

In [4]:
import pandas as pd
from rapidfuzz import process, fuzz
from utils.flights import AIRLINES
import ast


# Extract flight number (spoken digits to numerals)
def extract_flight_number(text):
    words = text.split()
    converted_words = [spoken_to_digit.get(word.lower(), word) for word in words]
    return "".join([char for char in converted_words if char.isdigit()]) or None


# Extract airline using fuzzy matching
def extract_airline(request_text):
    matches = process.extract(
        request_text, AIRLINES, scorer=fuzz.partial_ratio, limit=5
    )
    return max(matches, key=lambda x: x[1])[0] if matches else None


# Extract full flight name from request
def extract_flight_from_request(request_text, candidates):
    airline = extract_airline(request_text)
    flight_number = extract_flight_number(request_text)
    if not airline or not flight_number:
        return []
    query = f"{airline} {flight_number}"
    matches = process.extract(query, candidates, scorer=fuzz.ratio, limit=5)
    return [match[0] for match in matches]


# Evaluation metrics
def evaluate(predictions, ground_truth):
    top1, top3, reciprocal_ranks = 0, 0, []

    for preds, true_flight in zip(predictions, ground_truth):
        if not preds:
            reciprocal_ranks.append(0)
            continue

        try:
            rank = preds.index(true_flight) + 1
        except ValueError:
            rank = 0

        if rank == 1:
            top1 += 1
        if 1 <= rank <= 3:
            top3 += 1
        reciprocal_ranks.append(1 / rank if rank else 0)

    n = len(ground_truth)
    return {
        "Top-1 Accuracy": round(top1 / n, 3),
        "Top-3 Accuracy": round(top3 / n, 3),
        "MRR": round(sum(reciprocal_ranks) / n, 3),
    }


# Load dataset
df = pd.read_csv("data/dataset_final_test.csv")

# Ensure Candidates is parsed from string to list
df["Candidates"] = df["Candidates"].apply(ast.literal_eval)

# Extract predictions
all_predictions = []
for _, row in df.iterrows():
    preds = extract_flight_from_request(row["Request"], row["Candidates"])
    all_predictions.append(preds)

# Evaluate
results = evaluate(all_predictions, df["Flight"].tolist())
print("Evaluation Results:")
for k, v in results.items():
    print(f"{k}: {v}")

Evaluation Results:
Top-1 Accuracy: 0.312
Top-3 Accuracy: 0.738
MRR: 0.559
